In [1]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)
from rexpand_pyutils_file import write_file

from utils.conversation_data import load_conversation_data
from utils.json import to_json_compatible
from models.context import Context
from nodes.orchestrator import orchestrate
from models.workflow import State


INFO:root:LLM_MODEL: gpt-4.1-mini
INFO:root:LLM_TEMPERATURE: 0.0
INFO:root:LLM_USE_CACHE: False
INFO:root:LLM_INCLUDE_DEBUG_FIELDS: False
INFO:root:Read JSON data from ./input/categories.json
INFO:root:Read JSON data from ./input/categories.json


In [2]:
# Load all conversations
conversations = load_conversation_data('./input/convo_2454_rows.xlsx')


INFO:root:Read sheet 'Result 1' from Excel file ./input/convo_2454_rows.xlsx
INFO:root:Read EXCEL data from ./input/convo_2454_rows.xlsx with 2454 rows


In [3]:
# Select a conversation to process
index = 4
messages_end = 3

# In this case, the referrer asks some questions.
messages = conversations[index][:messages_end]
context = Context(messages=messages)
write_file('./output/current_messages.json', to_json_compatible(messages))

state = State(context=context)
write_file('./output/state.json', { "state": to_json_compatible(state.model_dump())})


INFO:root:Wrote JSON data to ./output/current_messages.json
INFO:root:Wrote JSON data to ./output/state.json


In [4]:
# First orchestrate: classify + summarize questions, then pause for user
state = orchestrate(state)

print(state.step)
print(state.classified_category)
print(state.required_actions)
print(state.fulfilled_actions)
print(state.suggested_topics)
print(state.generated_reply_message)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


todo: finish the workflow
ClassifierResult(category='request_additional_info', confidence=None, reason=None, referenced_message_ids=['e8a7f0a6-8007-4872-addc-9a467ffdd8d6'])
None
None
None


In [ ]:
from models.llm_result import FulfilledAction, FulfilledActionsResult

# User answers each summarized question (responses are example data for Patty)
ANSWERS = {
    "Answer question about the role": (
        "I'm most interested in the Data Analyst role on the analytics team, "
        "focusing on client reporting and SQL-based insights."
    ),
    "Answer question about target role": (
        "I'm targeting the Data Analyst role at Capco and would love to learn "
        "more about day-to-day responsibilities and team structure."
    ),
}

state.fulfilled_actions = FulfilledActionsResult(
    actions=[
        FulfilledAction(
            action=req.action,
            response=ANSWERS.get(
                req.action,
                f"Answer: {req.details or req.action}",
            ),
        )
        for req in state.required_actions.actions
    ]
)

# Second orchestrate: topics + reply message
state = orchestrate(state)

print(state.step)
print(state.classified_category)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)

In [ ]:
print(state.generated_reply_message.message)